In [0]:
opp_df = spark.read.table("01_bronze_catalog.raw_schema.opportunity")
opp_df.show(5)
opp_df.printSchema()
opp_df.count()

In [0]:

# opp_df = opp_df.fillna(999999,subset=['opportunity_id'])
# opp_df.count()

In [0]:
# bringing revenue_amount to double format
from pyspark.sql.functions import regexp_replace
opp_df = opp_df.withColumn("revenue_amount",regexp_replace(col("revenue_amount"), r"[^0-9.]", ""))
opp_df = opp_df.withColumn("revenue_amount",col("revenue_amount").cast("double"))
opp_df.show()


In [0]:
from pyspark.sql.functions import when,col,datediff

opp_df = opp_df.withColumn("contract_term",when(col("contract_term").isNull(),when(datediff(col("end_date"),col("start_date")) >= 364,"Yearly").otherwise("Monthly")).otherwise(col("contract_term")))

In [0]:
opp_df = opp_df.fillna("unknown",subset=["close_status"])

In [0]:
opp_df.write.mode("overwrite").saveAsTable("02_silver_catalog.transformed_schema.opportunity")

In [0]:
fx_df = spark.read.table("01_bronze_catalog.raw_schema.fx_rate")
fx_df.printSchema()
fx_df.show(5)

In [0]:
fx_df.write.mode("overwrite").saveAsTable("02_silver_catalog.transformed_schema.fx_rate")